In [ ]:
from rdkit.ML.Descriptors import MoleculeDescriptors
from contextlib import contextmanager
from rdkit.Chem import Descriptors
from typing import Generator
from pathlib import Path
from rdkit import Chem
import pandas as pd
import gzip
import json
import os
import re

make dict of the properties that are returned from QFP

In [40]:
with open("../data/properties.json", "r") as f:
    properties_list = json.load(f)

PROPERTY_DICT = {prop["property_db_id"]: prop["name_of_property"] for prop in properties_list}

In [62]:
PROPERTY_DICT

{0: 'energy',
 1: 'atomization_energy',
 2: 'homo_lumo_gap',
 3: 'ionization_energy',
 4: 'electron_affinity',
 5: 'chemical_potential',
 6: 'molecular_dipole',
 7: 'molecular_dipole_norm',
 8: 'molecular_quadrupole',
 9: 'molecular_quadrupole_principal_invariant_2',
 10: 'molecular_quadrupole_principal_invariant_3',
 11: 'molecular_polarizability',
 12: 'molecular_polarizability_mean',
 13: 'molecular_polarizability_anisotropy',
 14: 'normal_modes',
 15: 'normal_mode_frequencies',
 16: 'infrared_intensity',
 17: 'enthalpy',
 18: 'gibbs_free_energy',
 19: 'heat_capacity',
 20: 'entropy',
 21: 'zero_point_energy',
 22: 'radius_of_gyration',
 23: 'molecular_volume',
 24: 'molecular_sasa',
 25: 'atomic_sasa',
 26: 'effective_coordination_number',
 27: 'partial_charge',
 28: 'atomic_fukui_minus',
 29: 'atomic_fukui_plus',
 30: 'atomic_dipole',
 31: 'atomic_dipole_norm',
 32: 'atomic_quadrupole',
 33: 'atomic_quadrupole_principal_invariant_2',
 34: 'atomic_quadrupole_principal_invariant_3',

The output files of the QFP program are given in the following format:

"example_input_{id}.json" provides a **list** of conformers. For each conformer, the xyz coordinates + bunch of properties

In [ ]:
OUTPUT_PATH = Path("../data/QuantumFP/QFP_output")

def complete_path(path): return OUTPUT_PATH / Path(path)

# get QFP directory path
output_files: list[Path] = [complete_path(path) for path in os.listdir(OUTPUT_PATH) if path.endswith(".gz")]

print(len(output_files))

8837


In [45]:
descriptor_names = [name for name, _ in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

In [61]:
# for output_file in output_files:
with load_json_as_dataframe(output_files[0]) as df:
    print(df["nuclear_repulsion"].iloc[0])

[[1, 2, 11.123620003122184], [1, 3, 6.81794641409709], [1, 4, 3.893322911436289], [1, 5, 2.81849646069162], [1, 6, 2.3203836482158406], [1, 7, 2.3016384347497727], [1, 8, 2.790482824500981], [1, 9, 2.5645147130640047], [1, 10, 2.9199095870814418], [1, 11, 4.140529497040086], [1, 12, 5.491427017490376], [1, 13, 3.849633779101986], [1, 14, 0.8551553806727393], [1, 15, 0.42891484371996175], [1, 16, 0.32975366228195896], [1, 17, 0.32677371382072357], [1, 18, 0.3545216823013088], [1, 19, 0.41687288253153487], [1, 20, 0.7001136855128649], [2, 3, 8.463557339101543], [2, 4, 4.3819947239616255], [2, 5, 2.759951259303905], [2, 6, 2.190207898834915], [2, 7, 2.1742619692872243], [2, 8, 2.7509286661780163], [2, 9, 2.3741296224747845], [2, 10, 2.623217079396427], [2, 11, 3.809988801471785], [2, 12, 6.783347099626259], [2, 13, 4.370394801961319], [2, 14, 0.7961630679940126], [2, 15, 0.3971583321750454], [2, 16, 0.2989238133844882], [2, 17, 0.29678150071789744], [2, 18, 0.3155345118931659], [2, 19, 0.

In [ ]:
# for each file, load the file and convert it to a df
# dataframes: list[pd.DataFrame] = parallelize(process_molecule_data, output_files[:20])

Now, after processing all molecules, we need to do a thermal averaging over the different conformers (microstates) to represent the mixture of conformers that is present in reality. 

$$
<A> = \frac{\sum_i A_i e^{-E_i \beta}}{\sum_i e^{-E_i \beta}}
$$

In [60]:
atomic_features = {
    "effective_coordination_number",
    "partial_charge",
    "atomic_fukui_minus",
    "atomic_fukui_plus",
    "atomic_dipole_norm",
    "atomic_quadrupole_principal_invariant_2",
    "atomic_quadrupole_principal_invariant_3",
    "atomic_polarizability_mean",
    "atomic_polarizability_anisotropy",
    "percentage_buried_volume",
    "atomic_sasa",
    "partial_charge_water",
    "partial_charge_thf",
    "partial_charge_cyclohexane",
    "partial_charge_dmso",
    "atomic_charge_dipole_interaction",
    "atomic_charge_quadrupole_interaction",
    "atomic_dipole_dipole_interaction"
}
len(atomic_features)

18

add topological RDKit features

In [ ]:
# mol = Chem.MolFromSmiles(df["original_smiles"])

# descriptors = calc.CalcDescriptors(mol)
# print(descriptors)